In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# 1. Load the processed feature matrix
input_path = '../data/processed/feature_matrix.csv'
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Feature matrix not found at {input_path}. Run notebook 04 first.")

df = pd.read_csv(input_path)

# 2. Extract feature columns for scaling (exclude source_id)
feature_cols = [
    'absolute_magnitude_g',
    'bp_rp',
    'proper_motion_total',
    'tangential_velocity',
    'parallax_snr'
]

X = df[feature_cols].values

# 3. Standardize features (Mean = 0, Std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Initialize and fit baseline anomaly detection algorithms
print("Fitting Isolation Forest...")
iforest = IsolationForest(n_estimators=150, contamination=0.01, random_state=42)
# Raw decision function: lower/negative values indicate anomalies
if_raw_scores = -iforest.fit(X_scaled).decision_function(X_scaled)

print("Fitting Local Outlier Factor...")
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
# Raw negative outlier factor: higher values indicate stronger anomalies
lof_raw_scores = -lof.fit_predict(X_scaled) * lof.negative_outlier_factor_

print("Fitting One-Class SVM...")
ocsvm = OneClassSVM(nu=0.01, kernel='rbf', gamma='scale')
ocsvm.fit(X_scaled)
ocsvm_raw_scores = -ocsvm.decision_function(X_scaled)

# 5. Min-Max normalization to map scores to a unified [0, 1] scale
def min_max_scale(scores):
    return (scores - np.min(scores)) / (np.max(scores) - np.min(scores))

df['score_iforest'] = min_max_scale(if_raw_scores)
df['score_lof'] = min_max_scale(lof_raw_scores)
df['score_ocsvm'] = min_max_scale(ocsvm_raw_scores)

# 6. Consensus Anomaly Score (Equal weighting across models)
df['consensus_score'] = (
    df['score_iforest'] + 
    df['score_lof'] + 
    df['score_ocsvm']
) / 3.0

# 7. Rank candidates by consensus score
df_ranked = df.sort_values(by='consensus_score', ascending=False).reset_index(drop=True)

# 8. Save results
output_dir = '../results/candidates'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'baseline_anomaly_candidates.csv')
df_ranked.to_csv(output_path, index=False)

print(f"Baseline anomaly scoring complete. Ranked candidates saved to {output_path}")
print("\nTop 10 Anomaly Candidates:")
display(df_ranked[['source_id', 'consensus_score', 'score_iforest', 'score_lof', 'score_ocsvm'] + feature_cols].head(10))

Fitting Isolation Forest...
Fitting Local Outlier Factor...
Fitting One-Class SVM...
Baseline anomaly scoring complete. Ranked candidates saved to ../results/candidates/baseline_anomaly_candidates.csv

Top 10 Anomaly Candidates:


,source_id,consensus_score,score_iforest,score_lof,score_ocsvm,absolute_magnitude_g,bp_rp,proper_motion_total,tangential_velocity,parallax_snr
0,142031617837925376,0.961136,0.978579,0.981313,0.923515,12.882478,3.420682,282.792090,56.861855,411.413226
1,142883567549568896,0.950646,0.897267,0.985297,0.969373,14.931755,3.837551,300.609138,96.674740,33.986136
2,143170918043471744,0.948262,0.923627,0.996268,0.924889,5.263345,0.887662,204.347454,259.352132,204.049603
3,143438855283093888,0.939426,0.932197,0.968358,0.917724,8.844202,2.158354,152.629139,25.247240,1047.243766
4,139321149876099456,0.926101,0.828721,0.994656,0.954924,15.446749,1.322136,314.837037,135.119981,12.228148
5,142955551202830848,0.922338,0.935568,0.953814,0.877634,11.258115,2.885070,276.614984,95.541588,331.271793
6,143076256963396480,0.919756,0.916645,0.978606,0.864019,13.611241,0.610893,131.964435,23.161823,353.694054
7,143385185371780352,0.918114,0.882183,0.949487,0.922672,14.307637,0.657957,224.890450,45.229883,236.289240
8,141481037390861440,0.910139,0.923034,0.950766,0.856617,8.901859,1.452276,228.667812,102.533446,452.612510
9,138999542725203328,0.899165,0.941623,0.950926,0.804945,4.461815,0.768759,171.286144,33.782358,908.052990
